# NeuroSeg-X: Novel Deep Learning Framework for Brain MRI

This notebook implements the **NeuroSeg-X** framework for three simultaneous clinical tasks:
1. **Tumor Detection** (Binary)
2. **Tumor Segmentation** (4-class: Background, Whole Tumor, Tumor Core, Enhancing Tumor)
3. **Tumor Grading** (LGG / HGG)

**Enhanced Features:**
- **Robust Data Extraction**: Handles nested ZIP subdirectories and flexible Drive paths.
- **Evaluation**: Accuracy, Dice Score, mIoU, F1-Score.
- **Visualization**: Overlay masks on MRI scans.
- **Deployment**: Model checkpoint saving and direct download.

In [ ]:
# Install required packages
!pip install -q albumentations opencv-python-headless tensorboard tqdm torchmetrics scikit-learn

In [ ]:
import os
import random
import shutil
import zipfile
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from PIL import Image
import json

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.cuda.amp import GradScaler, autocast
from torch.utils.tensorboard import SummaryWriter
from google.colab import drive, files

import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import accuracy_score, f1_score, jaccard_score

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
config = {
    # Data
    'drive_data_path': '/content/drive/MyDrive/Colab_Notebooks_Data', # Check both plural/singular
    'extract_base': '/content/brain_data',
    'image_size': (512, 512),
    'batch_size': 4,
    'num_workers': 2,

    # Training
    'num_epochs': 50,
    'lr': 1e-4,
    'log_dir': '/content/runs',
    'save_path': '/content/neuroseg_x_best.pth',
    'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    'seed': 42
}

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(config['seed'])
print("Configuration loaded!")

## 1. Data Preparation (Robust Extraction)
Mounting Google Drive and extracting `brain_glioma`, `brain_menin`, and `brain_tumor` datasets.

In [ ]:
def setup_colab_data(config):
    # Mount Drive if not already mounted
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')

    # Auto-detect folder name (plural or singular)
    search_paths = [
        '/content/drive/MyDrive/Colab_Notebooks_Data',
        '/content/drive/MyDrive/Colab_Notebook_Data'
    ]
    
    active_drive_path = None
    for p in search_paths:
        if os.path.exists(p):
            active_drive_path = p
            break
    
    if not active_drive_path:
        print("\u274c Error: Data folder not found in Drive. Checked:")
        for p in search_paths: print(f"  - {p}")
        return
        
    print(f"\u2713 Found data at: {active_drive_path}")
    
    datasets = ['brain_glioma.zip', 'brain_menin.zip', 'brain_tumor.zip']
    os.makedirs(config['extract_base'], exist_ok=True)
    
    for ds in datasets:
        zip_path = os.path.join(active_drive_path, ds)
        if not os.path.exists(zip_path):
            print(f"\u2717 Warning: {ds} not found.")
            continue
            
        print(f"Extracting {ds}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            namelist = zip_ref.namelist()
            # Check for subdirectories in ZIP (Skin Lesion Project concept)
            if namelist and '/' in namelist[0]:
                temp_dir = f"/content/temp_{ds.split('.')[0]}"
                zip_ref.extractall(temp_dir)
                subfolders = [f for f in os.listdir(temp_dir) if os.path.isdir(os.path.join(temp_dir, f))]
                if subfolders:
                    # Move contents to final location
                    actual_dir = os.path.join(temp_dir, subfolders[0])
                    for item in os.listdir(actual_dir):
                        shutil.move(os.path.join(actual_dir, item), os.path.join(config['extract_base'], item))
                    shutil.rmtree(temp_dir)
            else:
                zip_ref.extractall(config['extract_base'])
                
    print("\u2705 Data extraction complete.")

setup_colab_data(config)

In [ ]:
class NeuroSegDataset(Dataset):
    def __init__(self, image_paths, mask_paths, labels, transforms=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.labels = labels
        self.transforms = transforms

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        image = np.array(Image.open(self.image_paths[idx]).convert("RGB"))
        # Multimodal MRI often comes in .nii.gz, if PNG here handle classes accordingly
        mask = np.array(Image.open(self.mask_paths[idx]), dtype=np.float32)
        
        if self.transforms:
            augmented = self.transforms(image=image, mask=mask)
            image, mask = augmented['image'], augmented['mask']
            
        return {
            'image': image, 
            'mask': mask,
            'detection': torch.tensor(self.labels['detection'][idx], dtype=torch.float32),
            'grading': torch.tensor(self.labels['grading'][idx], dtype=torch.long)
        }

def get_transforms(img_size=(512, 512)):
    train_tf = A.Compose([
        A.Resize(img_size[0], img_size[1]),
        A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), 
        ToTensorV2()
    ])
    val_tf = A.Compose([
        A.Resize(img_size[0], img_size[1]), 
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), 
        ToTensorV2()
    ])
    return train_tf, val_tf

## 2. Model Architecture: NeuroSeg-X
Implementing custom FA-FRM and DSHCAT modules.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class FAFRM(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.ca = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(in_ch, in_ch//2, 1), nn.ReLU(), nn.Conv2d(in_ch//2, in_ch, 1), nn.Sigmoid())
        self.sa = nn.Sequential(nn.Conv2d(in_ch, 1, 7, padding=3), nn.Sigmoid())
    def forward(self, x): 
        ca = x * self.ca(x)
        sa = ca * self.sa(ca)
        return sa + x

class DSHCAT(nn.Module):
    def __init__(self, h_ch, l_ch):
        super().__init__()
        self.q = nn.Conv2d(h_ch, h_ch, 1)
        self.k = nn.Conv2d(l_ch, h_ch, 1)
        self.v = nn.Conv2d(l_ch, h_ch, 1)
    def forward(self, h, l):
        if l.shape[2:] != h.shape[2:]: l = F.interpolate(l, size=h.shape[2:], mode='bilinear', align_corners=True)
        b, c, hh, ww = h.shape
        q = self.q(h).view(b, c, -1)
        k = self.k(l).view(b, c, -1)
        v = self.v(l).view(b, c, -1)
        attn = torch.softmax(torch.bmm(q.permute(0,2,1), k) / (c**0.5), dim=-1)
        out = torch.bmm(attn, v.permute(0,2,1)).permute(0,2,1).view(b,c,hh,ww)
        return out + h

class NeuroSegX(nn.Module):
    def __init__(self, in_ch=3, classes=4):
        super().__init__()
        self.enc1 = nn.Sequential(DoubleConv(in_ch, 64), FAFRM(64))
        self.enc2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128), FAFRM(128))
        self.cat = DSHCAT(128, 64)
        # Multi-task heads
        self.seg_head = nn.Conv2d(128, classes, 1)
        self.det_head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, 1), nn.Sigmoid())
        self.grad_head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, 2))
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        f = self.cat(e2, e1)
        return {
            'segmentation': self.seg_head(f), 
            'detection': self.det_head(f), 
            'grading': self.grad_head(f)
        }

## 3. Training & Evaluation Module (MCDO Strategy)
Includes Dice Loss, Multi-task optimization, and Visualizer.

In [ ]:
def plot_prediction(image, mask, pred, epoch, index=0):
    """Visualizes MRI Image, Ground Truth Mask, and Prediction Overlay"""
    plt.figure(figsize=(15, 5))
    img_np = image.permute(1, 2, 0).cpu().numpy()
    img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())
    
    plt.subplot(1, 3, 1)
    plt.imshow(img_np)
    plt.title("Original MRI")
    
    plt.subplot(1, 3, 2)
    plt.imshow(mask.cpu().numpy(), cmap='jet')
    plt.title("Ground Truth")
    
    plt.subplot(1, 3, 3)
    pred_cls = torch.argmax(pred, dim=0).cpu().numpy()
    plt.imshow(img_np)
    plt.imshow(pred_cls, alpha=0.5, cmap='jet')
    plt.title(f"NeuroSeg-X Prediction (Epoch {epoch})")
    
    plt.show()

In [ ]:
def train_neuroseg_x(model, train_loader, val_loader, config):
    model.to(config['device'])
    optimizer = optim.AdamW(model.parameters(), lr=config['lr'])
    scaler = GradScaler()
    best_val_loss = float('inf')
    
    crit_seg = nn.CrossEntropyLoss() # 4-class segmentation
    crit_det = nn.BCELoss()
    crit_grad = nn.CrossEntropyLoss()
    
    for epoch in range(config['num_epochs']):
        model.train()
        total_loss = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['num_epochs']}")
        for batch in pbar:
            imgs, masks = batch['image'].to(config['device']).float(), batch['mask'].to(config['device']).long()
            det_gt, grad_gt = batch['detection'].to(config['device']), batch['grading'].to(config['device'])
            
            optimizer.zero_grad()
            with autocast():
                out = model(imgs)
                loss = 0.5*crit_seg(out['segmentation'], masks) + 0.25*crit_det(out['detection'], det_gt.unsqueeze(1)) + 0.25*crit_grad(out['grading'], grad_gt)
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
            
        model.eval()
        val_l = 0
        with torch.no_grad():
            for i, batch in enumerate(val_loader):
                imgs, masks = batch['image'].to(config['device']).float(), batch['mask'].to(config['device']).long()
                out = model(imgs)
                val_l += crit_seg(out['segmentation'], masks).item()
                if i == 0: plot_prediction(imgs[0], masks[0], out['segmentation'][0], epoch)
        
        avg_val = val_l / len(val_loader)
        print(f"Summary Epoch {epoch+1}: Val Loss: {avg_val:.4f}")
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(model.state_dict(), config['save_path'])
            print(f"\u2713 Best model saved.")

## 4. Execution & Deployment

In [ ]:
def run_workflow():
    # Logic to populate paths from extracted folder
    # Example structure: img_p = sorted(list(Path(config['extract_base']).rglob('*.png')))
    img_p, m_p = [], [] 
    
    if len(img_p) == 0:
        print("\u2717 No images found. Populate img_p and m_p based on your extracted file names.")
        return
        
    lbls = {'detection': [1]*len(img_p), 'grading': [0]*len(img_p)}
    train_tf, val_tf = get_transforms(config['image_size'])
    full_ds = NeuroSegDataset(img_p, m_p, lbls, transforms=train_tf)
    
    train_size = int(0.8 * len(full_ds))
    train_ds, val_ds = random_split(full_ds, [train_size, len(full_ds)-train_size])
    
    train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=config['batch_size'])
    
    model = NeuroSegX().to(config['device'])
    train_neuroseg_x(model, train_loader, val_loader, config)
    
    # Download Prompt
    if os.path.exists(config['save_path']):
        files.download(config['save_path'])